# Nova AI — Training Notebook

**Run cells IN ORDER, top to bottom. Do NOT skip.**

1. Runtime → Change runtime type → **T4 GPU** (free)
2. Click the 🔑 key icon (left sidebar) → add secret `HF_TOKEN`
3. Run each cell one by one

Cell order:
```
1 → Check GPU
2 → Install + clone repo       (~2 min)
3 → Prepare data               (~5 min)
4 → Train Nova Chat            (~30-60 min)
5 → Train Nova Code            (~30-60 min)
6 → Test models                (run AFTER 4 and 5)
7 → Push to Hugging Face
8 → Download weights to PC
```

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
else:
    print('NO GPU FOUND.')
    print('Go to Runtime → Change runtime type → T4 GPU, then re-run.')
    raise SystemExit('Need GPU. Change runtime type first.')

In [ ]:
# ── Cell 2: Clone repo + install ───────────────────────────────────
import os

if not os.path.exists('Phollama-OS'):
    !git clone https://github.com/genity5-collab/Phollama-OS.git
else:
    print('Repo already cloned, pulling latest...')
    !cd Phollama-OS && git pull

%cd Phollama-OS

!pip install torch tokenizers datasets huggingface_hub numpy -q
print('\nDependencies installed.')
print('Working directory:', os.getcwd())

In [ ]:
# ── Cell 3: Prepare data + train tokenizer ─────────────────────────
# Downloads open datasets and trains your BPE tokenizer.
# Must complete before training.
import os
os.makedirs('data',        exist_ok=True)
os.makedirs('models',      exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

!python data/prepare.py

# Verify outputs exist
required = [
    'models/nova_tokenizer.json',
    'data/chat_train.bin',
    'data/chat_val.bin',
    'data/code_train.bin',
    'data/code_val.bin',
]
print('\nChecking outputs:')
all_ok = True
for f in required:
    exists = os.path.exists(f)
    size   = os.path.getsize(f) if exists else 0
    status = '✅' if exists and size > 0 else '❌'
    print(f'  {status} {f} ({size/1e6:.1f} MB)')
    if not exists or size == 0:
        all_ok = False

if all_ok:
    print('\n✅ Data ready. Proceed to Cell 4.')
else:
    print('\n❌ Some files missing. Check errors above before continuing.')

In [ ]:
# ── Cell 4: Train Nova Chat ────────────────────────────────────────
# ~30-60 min on free T4 GPU.
# Watch the loss numbers:
#   train loss falling + val loss falling = GOOD
#   val loss rising while train falls     = overfitting, will auto-stop

import os
os.makedirs('checkpoints/nova-chat', exist_ok=True)

print('Starting Nova Chat training...')
print('This will take 30-60 minutes. Watch the loss numbers.')
print('-' * 60)

!python src/train.py --model chat

# Verify checkpoint saved
best  = 'checkpoints/nova-chat/best.pt'
final = 'checkpoints/nova-chat/final.pt'
if os.path.exists(best):
    size = os.path.getsize(best) / 1e6
    print(f'\n✅ Nova Chat trained: best.pt ({size:.0f} MB)')
elif os.path.exists(final):
    size = os.path.getsize(final) / 1e6
    print(f'\n✅ Nova Chat trained: final.pt ({size:.0f} MB)')
else:
    print('\n❌ No checkpoint found. Check errors above.')

In [ ]:
# ── Cell 5: Train Nova Code ────────────────────────────────────────
# Same as Cell 4 but for the coding model.
# Run AFTER Cell 4 finishes.

import os
os.makedirs('checkpoints/nova-code', exist_ok=True)

print('Starting Nova Code training...')
print('-' * 60)

!python src/train.py --model code

best  = 'checkpoints/nova-code/best.pt'
final = 'checkpoints/nova-code/final.pt'
if os.path.exists(best):
    size = os.path.getsize(best) / 1e6
    print(f'\n✅ Nova Code trained: best.pt ({size:.0f} MB)')
elif os.path.exists(final):
    size = os.path.getsize(final) / 1e6
    print(f'\n✅ Nova Code trained: final.pt ({size:.0f} MB)')
else:
    print('\n❌ No checkpoint found. Check errors above.')

In [ ]:
# ── Cell 6: Test models ────────────────────────────────────────────
# RUN THIS ONLY AFTER CELLS 4 AND 5 FINISH.
# Tests both trained models with sample prompts.

import os
import sys
import torch
sys.path.insert(0, '.')

from src.model import NovaGPT
from src.tokenizer import NovaTokenizer
from src.config import NovaChatConfig, NovaCodeConfig

# Check checkpoints exist before loading
def find_checkpoint(model_type):
    best  = f'checkpoints/nova-{model_type}/best.pt'
    final = f'checkpoints/nova-{model_type}/final.pt'
    if os.path.exists(best):
        return best
    if os.path.exists(final):
        return final
    return None

chat_ckpt = find_checkpoint('chat')
code_ckpt = find_checkpoint('code')

if not chat_ckpt:
    raise FileNotFoundError('Nova Chat not trained yet. Run Cell 4 first.')
if not code_ckpt:
    raise FileNotFoundError('Nova Code not trained yet. Run Cell 5 first.')

print(f'Chat checkpoint: {chat_ckpt}')
print(f'Code checkpoint: {code_ckpt}')

# Load tokenizer
tokenizer = NovaTokenizer('models/nova_tokenizer.json')
print(f'Tokenizer vocab size: {tokenizer.vocab_size}')

def test_model(config, prompt, ckpt_path):
    model, ckpt = NovaGPT.load_checkpoint(ckpt_path, config)
    model.eval()
    print(f'  Loss: {ckpt.get("loss", "?"):.4f} | Step: {ckpt.get("step", "?")}')
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    idx = torch.tensor([ids])
    with torch.no_grad():
        out = model.generate(idx, max_new_tokens=120, temperature=0.8, top_k=40)
    new_ids  = out[0][len(ids):].tolist()
    response = tokenizer.decode(new_ids)
    for stop in ['<|user|>', '<|system|>', '<eos>', '<|assistant|>']:
        if stop in response:
            response = response[:response.index(stop)]
    return response.strip() or '(empty response - model needs more training data)'

# Test prompts
tests = [
    ('Nova Chat', NovaChatConfig, chat_ckpt,
     '<|system|>You are Nova.<|user|>What is your name?<|assistant|>'),
    ('Nova Chat', NovaChatConfig, chat_ckpt,
     '<|system|>You are Nova.<|user|>What is 5 plus 3?<|assistant|>'),
    ('Nova Code', NovaCodeConfig, code_ckpt,
     '<|system|>You are Nova Code.<|user|>Write hello world in Python.<|assistant|>'),
    ('Nova Code', NovaCodeConfig, code_ckpt,
     '<|system|>You are Nova Code.<|user|>Write a function to add two numbers.<|assistant|>'),
]

print('\n' + '='*60)
for name, config, ckpt, prompt in tests:
    user_q = prompt.split('<|user|>')[1].split('<|assistant|>')[0].strip()
    print(f'\n{name}: {user_q}')
    response = test_model(config, prompt, ckpt)
    print(f'  Response: {response[:200]}')
    print('-'*60)

print('\nTest complete.')
print('If responses look like random text: model needs more training data.')
print('If responses look coherent: proceed to Cell 7 to push to HuggingFace.')

In [ ]:
# ── Cell 7: Run eval score ─────────────────────────────────────────
# Scores the model on 20 test cases each.
# Score 75+ = good to deploy. Under 50 = needs more data.

print('Evaluating Nova Chat...')
!python eval/evaluate.py --model chat
print('\nEvaluating Nova Code...')
!python eval/evaluate.py --model code

In [ ]:
# ── Cell 8: Push to Hugging Face ───────────────────────────────────
# CHANGE YOUR USERNAME BELOW
YOUR_HF_USERNAME = 'genity5-collab'  # ← change this if different

import os
from google.colab import userdata
from huggingface_hub import login, HfApi

login(userdata.get('HF_TOKEN'))
api = HfApi()

files_to_upload = [
    ('models/nova_tokenizer.json', 'tokenizer.json'),
    ('src/model.py',               'src/model.py'),
    ('src/config.py',              'src/config.py'),
    ('src/tokenizer.py',           'src/tokenizer.py'),
    ('src/generate.py',            'src/generate.py'),
]

for model_type in ['chat', 'code']:
    repo_id = f'{YOUR_HF_USERNAME}/nova-{model_type}-124m'
    best    = f'checkpoints/nova-{model_type}/best.pt'
    final   = f'checkpoints/nova-{model_type}/final.pt'
    ckpt    = best if os.path.exists(best) else final

    if not os.path.exists(ckpt):
        print(f'❌ Skipping {model_type}: no checkpoint found')
        continue

    print(f'\nPushing Nova {model_type} to {repo_id}...')
    api.create_repo(repo_id=repo_id, exist_ok=True)

    api.upload_file(path_or_fileobj=ckpt, path_in_repo='model.pt', repo_id=repo_id)
    print(f'  ✅ model.pt')

    for local, remote in files_to_upload:
        if os.path.exists(local):
            api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=repo_id)
            print(f'  ✅ {remote}')

    print(f'  🌐 https://huggingface.co/{repo_id}')

print('\n✅ All pushed to Hugging Face.')

In [ ]:
# ── Cell 9: Download to your PC ────────────────────────────────────
# Downloads trained weights to your computer.
# Save them to the right folders on your PC.

import os
from google.colab import files

to_download = [
    ('checkpoints/nova-chat/best.pt',  'checkpoints/nova-chat/final.pt',  'nova-chat-best.pt'),
    ('checkpoints/nova-code/best.pt',  'checkpoints/nova-code/final.pt',  'nova-code-best.pt'),
    ('models/nova_tokenizer.json',     None,                               'nova_tokenizer.json'),
]

for primary, fallback, save_name in to_download:
    path = primary if os.path.exists(primary) else (fallback if fallback and os.path.exists(fallback) else None)
    if path:
        size = os.path.getsize(path) / 1e6
        print(f'Downloading {save_name} ({size:.0f} MB)...')
        files.download(path)
    else:
        print(f'❌ {save_name} not found')

print('\nSave downloaded files to your PC:')
print('  nova-chat-best.pt  →  checkpoints/nova-chat/best.pt')
print('  nova-code-best.pt  →  checkpoints/nova-code/best.pt')
print('  nova_tokenizer.json → models/nova_tokenizer.json')
print('\nThen run locally:')
print('  python src/generate.py --model chat')
print('  python src/generate.py --model code')